# Scientific comparison: Skill Memory vs. established OCL strategies

This notebook is the analysis layer for the reproducible benchmark workflow. It does **not** manufacture toy results and it does not rerun only Skill Memory. The workflow runs the same benchmark protocol for every strategy and five independent seeds, then this notebook aggregates those artifacts.

**Protocol:** Split CIFAR-100, 20 class-incremental experiences, memory size 2000, five seeds (0--4), Python 3.11, Avalanche 0.6.0. Established strategy implementations come from the pinned OCL Survey revision used by the workflow; Skill Memory is the Avalanche plugin in this repository.

The primary comparison is **active-model** final test accuracy and causal average forgetting. Skill Memory retrieval is intentionally not used for the primary table because `before_eval_exp` uses labeled evaluation data and therefore answers a separate oracle-routing question.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'results').exists():
    REPO_ROOT = REPO_ROOT.parent
RESULTS = REPO_ROOT / 'results'
EXPECTED_SEEDS = {0,1,2,3,4}
STRATEGIES = ['er','er_ace','der','mir','er_lwf','rar','scr','agem','mer','icarl','gdumb','skill_memory']
LABELS = {'er':'ER','er_ace':'ER-ACE','der':'DER++','mir':'MIR','er_lwf':'ER + LwF','rar':'RAR','scr':'SCR','agem':'A-GEM','mer':'MER','icarl':'iCaRL','gdumb':'GDumb','skill_memory':'Skill Memory'}
print('Results root:', RESULTS)


## 1. Load all available seed artifacts

Every method is evaluated from its own workflow output. A missing seed is reported explicitly rather than silently replacing it with a different seed or an old CSV.


In [ ]:
def json_objects(root):
    objects=[]
    for path in root.rglob('*.json'):
        try:
            text=path.read_text()
        except OSError:
            continue
        # OCL Survey writes JSON Lines; Skill Memory writes one JSON object.
        for line in text.splitlines():
            line=line.strip()
            if not line: continue
            try: objects.append(json.loads(line))
            except json.JSONDecodeError: continue
    return objects

def causal_forgetting(rows):
    task_cols=[f'Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp{i:03d}' for i in range(20)]
    values=[]
    for col in task_cols:
        seq=[float(r[col]) for r in rows if col in r and r[col] is not None]
        if len(seq)>1:
            values.append(max(seq[:-1])-seq[-1])
    return float(np.mean(values)) if values else np.nan

def load_ocl(method):
    root=RESULTS / f'{method}_split_cifar100_20_2000'
    if not root.exists(): return pd.DataFrame()
    rows=[]
    for seed_dir in sorted(root.iterdir()):
        if not seed_dir.is_dir() or not seed_dir.name.isdigit(): continue
        seed=int(seed_dir.name)
        objs=json_objects(seed_dir)
        metric='Top1_Acc_Stream/eval_phase/test_stream/Task000'
        valid=[r for r in objs if metric in r and r[metric] is not None and 'mb_index' in r]
        valid.sort(key=lambda r:r['mb_index'])
        if not valid: continue
        last=valid[-1]
        rows.append({'method':LABELS[method],'seed':seed,'final_accuracy':float(last[metric]),'forgetting':causal_forgetting(valid)})
    return pd.DataFrame(rows)

def load_skill_memory():
    rows=[]
    root=RESULTS/'skill_memory_split_cifar100'
    if not root.exists(): return pd.DataFrame()
    for path in sorted(root.glob('*/summary.json')):
        data=json.loads(path.read_text())
        rows.append({'method':'Skill Memory','seed':int(data['seed']),'final_accuracy':float(data['final_accuracy']),'forgetting':float(data['forgetting']),'AAA_test':float(data.get('AAA_test',np.nan))})
    return pd.DataFrame(rows)

frames=[load_ocl(m) for m in STRATEGIES if m!='skill_memory']+[load_skill_memory()]
per_seed=pd.concat([f for f in frames if not f.empty],ignore_index=True) if any(not f.empty for f in frames) else pd.DataFrame()
if per_seed.empty: raise RuntimeError('No scientific result artifacts found. Run .github/workflows/scientific-comparison.yml first.')
coverage=(per_seed.groupby('method')['seed'].apply(lambda s:sorted(set(s))).rename('seeds').reset_index())
coverage['complete']=coverage['seeds'].apply(lambda s:set(s)==EXPECTED_SEEDS)
display(coverage)


## 2. Five-seed scientific summary

Means and standard deviations are computed across seeds, not across individual task observations. Only methods with all five required seeds enter the primary table.


In [ ]:
complete_methods=coverage.loc[coverage['complete'],'method'].tolist()
complete=per_seed[per_seed['method'].isin(complete_methods)].copy()
summary=(complete.groupby('method').agg(n=('seed','nunique'),final_accuracy=('final_accuracy','mean'),final_accuracy_std=('final_accuracy','std'),forgetting=('forgetting','mean'),forgetting_std=('forgetting','std')).sort_values(['final_accuracy','forgetting'],ascending=[False,True]))
summary[['final_accuracy','final_accuracy_std','forgetting','forgetting_std']]*=100
display(summary.style.format({'final_accuracy':'{:.2f}%','final_accuracy_std':'{:.2f}%','forgetting':'{:.2f}%','forgetting_std':'{:.2f}%'}))


## 3. Per-seed values and Skill Memory decisions

A mean can hide seed sensitivity. Skill Memory also exposes REUSE/CLONE/SCRATCH decisions and compatibility diagnostics so its behavior is auditable.


In [ ]:
display(complete.sort_values(['method','seed']).reset_index(drop=True))
decision_rows=[]
root=RESULTS/'skill_memory_split_cifar100'
for path in sorted(root.glob('*/summary.json')) if root.exists() else []:
    data=json.loads(path.read_text())
    for d in data.get('decisions',[]): decision_rows.append({'seed':data['seed'],**d})
decisions=pd.DataFrame(decision_rows)
if not decisions.empty:
    display(pd.crosstab(decisions['seed'],decisions['decision'],margins=True))
    display(decisions.groupby('decision')[['compatibility_score','old_accuracy','new_accuracy']].mean())


## 4. Figures

Error bars are seed standard deviations. These plots use the same primary metrics as the table and do not mix oracle retrieval accuracy with ordinary continual-learning accuracy.


In [ ]:
plot_df=summary.reset_index()
plt.figure(figsize=(11,5))
plt.errorbar(plot_df['method'],plot_df['final_accuracy'],yerr=plot_df['final_accuracy_std'],fmt='o')
plt.ylabel('Final test accuracy (%)'); plt.xlabel('Strategy'); plt.xticks(rotation=45,ha='right'); plt.title('Split CIFAR-100: final accuracy across five seeds'); plt.tight_layout(); plt.show()
plt.figure(figsize=(11,5))
plt.errorbar(plot_df['method'],plot_df['forgetting'],yerr=plot_df['forgetting_std'],fmt='o')
plt.ylabel('Average forgetting (%)'); plt.xlabel('Strategy'); plt.xticks(rotation=45,ha='right'); plt.title('Split CIFAR-100: causal average forgetting across five seeds'); plt.tight_layout(); plt.show()


## Interpretation guardrails

- A method is not scientifically comparable here until all five seeds are present.
- Skill Memory's primary result is active-model retention; labeled retrieval is a separate experiment.
- The workflow pins the OCL Survey revision so reference implementations are reproducible.
- No superiority claim is encoded in the notebook; conclusions should follow the observed replicate distributions.
